<a href="https://colab.research.google.com/github/zvtech89/CAI-Class/blob/main/football_shoes_beautifulsoup_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Scraping Football Shoes with Python and BeautifulSoup

**Objective:** Browse an e-commerce website, filter football shoes/cleats, and collect these attributes:

1. Product Image  
2. Brand  
3. Price  
4. Customer Rating  


## Ethical considerations for web scraping

Before collecting data from any website, it is important to scrape responsibly:

1. **Check the website Terms of Service** and avoid scraping content that the site does not allow.
2. **Review robots.txt** to understand which parts of the website are restricted for bots.
3. **Do not overload the server.** Use delays between requests and avoid aggressive scraping.
4. **Collect only the data needed** for the assignment. Avoid personal, private, or sensitive data.
5. **Respect copyright and ownership.** Product images and descriptions may belong to the retailer or brand.
6. **Identify your request clearly** with a normal user-agent and avoid bypassing security or anti-bot protections.
7. **Use the data for educational purposes only** unless you have permission for commercial use.

In [ ]:
# Install/import required libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import json
from urllib.parse import urljoin
from IPython.display import display, Image

In [ ]:
# Website URL
# You can replace this URL with a search/category page from a retailer that lists football shoes or football cleats.
# Example: a football boots/category page from a sporting goods website.
url = "https://www.sportsdirect.com/football/football-boots"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

response = requests.get(url, headers=headers, timeout=20)
print("Status code:", response.status_code)
print("Page length:", len(response.text))

soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
# Helper functions
football_keywords = [
    "football", "cleat", "cleats", "boot", "boots", "soccer", "firm ground", "fg", "turf", "indoor"
]

def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def contains_football_keyword(text):
    text = clean_text(text).lower()
    return any(keyword in text for keyword in football_keywords)


def extract_price(text):
    text = clean_text(text)
    match = re.search(r"(\$|£|€)\s?\d+(?:[,.]\d{2})?", text)
    return match.group(0) if match else "Not listed"


def extract_rating(text):
    text = clean_text(text)
    # Common rating formats: 4.5, 4.5 out of 5, Rating: 4.5
    match = re.search(r"([0-5](?:\.\d)?)\s*(?:out of\s*5|/\s*5|stars?)?", text, flags=re.I)
    return match.group(1) if match else "Not listed"


def guess_brand(product_name):
    known_brands = [
        "Nike", "Adidas", "Puma", "Under Armour", "New Balance", "Mizuno", "Umbro", "Kipsta", "Diadora", "Skechers"
    ]
    for brand in known_brands:
        if brand.lower() in product_name.lower():
            return brand
    # Fallback: first word as possible brand
    return product_name.split()[0] if product_name else "Not listed"

In [ ]:
# Method 1: Try to scrape product information from JSON-LD structured data
# Many e-commerce sites store product data in <script type="application/ld+json">.

products = []

for script in soup.find_all("script", type="application/ld+json"):
    try:
        data = json.loads(script.string or "{}")
    except Exception:
        continue

    items = []
    if isinstance(data, dict):
        if data.get("@type") == "Product":
            items = [data]
        elif "itemListElement" in data:
            for item in data.get("itemListElement", []):
                if isinstance(item, dict):
                    item_obj = item.get("item", item)
                    if isinstance(item_obj, dict):
                        items.append(item_obj)
    elif isinstance(data, list):
        items = [d for d in data if isinstance(d, dict) and d.get("@type") == "Product"]

    for item in items:
        name = clean_text(item.get("name"))
        if not contains_football_keyword(name):
            continue

        brand = item.get("brand", "")
        if isinstance(brand, dict):
            brand = brand.get("name", "")
        brand = clean_text(brand) or guess_brand(name)

        image = item.get("image", "")
        if isinstance(image, list):
            image = image[0] if image else ""
        image = urljoin(url, clean_text(image)) if image else "Not listed"

        offers = item.get("offers", {})
        if isinstance(offers, list):
            offers = offers[0] if offers else {}
        price = offers.get("price", "") if isinstance(offers, dict) else ""
        currency = offers.get("priceCurrency", "") if isinstance(offers, dict) else ""
        price = f"{currency} {price}".strip() if price else "Not listed"

        rating = item.get("aggregateRating", {})
        rating = rating.get("ratingValue", "Not listed") if isinstance(rating, dict) else "Not listed"

        products.append({
            "Product Name": name,
            "Product Image": image,
            "Brand": brand,
            "Price": price,
            "Customer Rating": rating
        })

print("Products found from JSON-LD:", len(products))

In [ ]:
# Method 2: Fallback scraper using product-card style HTML selectors
# This is useful when JSON-LD is not available.

if len(products) == 0:
    possible_cards = soup.select(
        "li, article, div.product, div.product-card, div.ProductCard, div.s-productthumbbox, div[class*='product'], div[class*='Product']"
    )

    seen = set()

    for card in possible_cards:
        card_text = clean_text(card.get_text(" "))
        if len(card_text) < 20 or not contains_football_keyword(card_text):
            continue

        # Product name
        name_tag = card.select_one("a[title], h2, h3, span[class*='name'], div[class*='name'], a")
        name = clean_text(name_tag.get("title") if name_tag and name_tag.has_attr("title") else name_tag.get_text(" ") if name_tag else "")
        if not name:
            name = card_text[:120]

        # Avoid duplicates
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)

        # Image
        img = card.select_one("img")
        image = "Not listed"
        if img:
            image_src = img.get("src") or img.get("data-src") or img.get("data-original") or img.get("data-lazy")
            if image_src:
                image = urljoin(url, image_src)

        brand = guess_brand(name)
        price = extract_price(card_text)

        # Rating from aria-label, alt text, or visible card text
        rating_sources = [card_text]
        for tag in card.select("[aria-label], [title], img[alt]"):
            rating_sources.append(tag.get("aria-label") or tag.get("title") or tag.get("alt") or "")
        rating = "Not listed"
        for source in rating_sources:
            if "star" in source.lower() or "rating" in source.lower() or "out of" in source.lower():
                rating = extract_rating(source)
                if rating != "Not listed":
                    break

        products.append({
            "Product Name": name,
            "Product Image": image,
            "Brand": brand,
            "Price": price,
            "Customer Rating": rating
        })

        time.sleep(0.1)

print("Total products found:", len(products))

In [ ]:
# Create final dataframe
football_shoes_df = pd.DataFrame(products)

# Keep only the required assignment attributes plus Product Name for readability
if not football_shoes_df.empty:
    football_shoes_df = football_shoes_df[["Product Name", "Product Image", "Brand", "Price", "Customer Rating"]]
    football_shoes_df = football_shoes_df.drop_duplicates(subset=["Product Name", "Brand", "Price"])

football_shoes_df.head(20)

In [ ]:
# Save the result to CSV
output_file = "football_shoes_scraped_data.csv"
football_shoes_df.to_csv(output_file, index=False)
print(f"Saved file: {output_file}")

# In Google Colab, this will download the CSV to your computer.
try:
    from google.colab import files
    files.download(output_file)
except Exception:
    print("Download works automatically in Google Colab. Outside Colab, check the current notebook folder.")

## Conclusion

This project used **BeautifulSoup** to collect football shoe product data from a website. The scraper looks for football-related keywords and extracts the required attributes: product image, brand, price, and customer rating. If structured JSON-LD product data is available, the notebook uses it first. If not, it falls back to parsing product cards from the webpage.